# 18 · One model, a model told the QP, or a specialist?

Notebook 17 found that the shipping model's gain over bicubic shrinks as QP rises, and is gone from QP 42. That model was
trained at one bitrate, so a frame at a high QP is new to it. Is it worth training for high QP, and how?

Three models, each with notebook 8's body (24 channels, 2 residual blocks) and recipe, on the pairs notebook 17 writes:

1. **Blind** - every QP, not told it.
2. **Told the QP** - every QP, with each frame's QP as a second input (`models.ToldLuma`).
3. **Specialist** - only frames at QP 38 and up, not told.

Each is scored on the validation pairs by QP bin, as a gain over bicubic. Separate models pay only if the specialist clearly
beats the told model at high QP.

**Limits.** The pairs are video only (notebook 17 takes its targets from YouTube videos), so this says nothing about
screens yet. Each model is trained once, at one seed: on notebook 8's set two seeds of one model landed 0.0002 SSIM-Y apart
(notebook 16), so a gap of that size here is luck.

Needs: notebook 17, part 5 (`qp_pairs/`). Writes each training to `checkpoints/qp_runs/`.

In [2]:
"""Settings."""

import os

import numpy as np
import torch
from torch.utils.data import DataLoader, Dataset, Subset

os.environ["DATASETS_DIR"] = "/mnt/d/datasets"   # where the datasets are: set before helpers is imported

from helpers import metrics, models, patches, sources, training
from helpers.training import DEVICE

PAIRS_DIR = sources.ROOT / "upscale" / "qp_pairs"   # notebook 17, part 5
RUN_DIR = training.CHECKPOINT_DIR / "qp_runs"       # each training, saved every epoch

QP_EDGES = (0, 18, 26, 30, 34, 38, 42, 46)          # notebook 17's bins of a frame's QP
SPECIALIST_FROM = 38                                # the specialist sees QP 38 and up
WIDTH, BLOCKS, STEM = 24, 2, 3                      # notebook 8's body
BATCH_SIZE, LEARNING_RATE, MAX_EPOCHS, SEED = 32, 1e-3, 500, 0

training.seed_everything(SEED)

## 1. Data

Every patch comes with its frame's QP, divided by 51 so it runs from 0 to 1. Every loader hands it out; the blind models
drop it.

In [3]:
manifest = training.load_manifest(PAIRS_DIR)
training.describe(manifest)
SCALE = manifest["scale"]
trim = training.trimmer(manifest)


class WithQP(Dataset):
    """A split's pairs as ((LR, QP / 51), HR)."""

    def __init__(self, split, augment):
        self.pairs = patches.PatchPairs(PAIRS_DIR, split, manifest, augment=augment)
        self.qp = np.load(PAIRS_DIR / manifest["splits"][split]["qp"])
        self.condition = torch.from_numpy(self.qp / 51).float()[:, None, None, None]

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, index):
        lr, hr = self.pairs[index]
        return (lr, self.condition[index]), hr


def batches(dataset, low=0, high=99, train=False):
    """A loader of `dataset`'s patches at QP from `low` up to `high`."""
    kept = Subset(dataset, np.flatnonzero((dataset.qp >= low) & (dataset.qp < high)))
    return DataLoader(kept, batch_size=BATCH_SIZE if train else 128, shuffle=train, drop_last=train)


train_set, val_set = WithQP("train", augment=True), WithQP("val", augment=False)
BINS = list(zip(QP_EDGES[:-1], QP_EDGES[1:]))
print(f"{'patches':<8}{'all':>8}" + "".join(f"{f'{low}-{high}':>8}" for low, high in BINS))
for name, dataset in (("train", train_set), ("val", val_set)):
    print(f"{name:<8}{len(dataset):8}" + "".join(f"{((dataset.qp >= low) & (dataset.qp < high)).sum():8}" for low, high in BINS))

device  cuda NVIDIA GeForce RTX 3090
scale   2
patch   72 px LR in (64 px + a halo of 4) -> 128 px HR out, 1 frames each
patches      all    0-18   18-26   26-30   30-34   34-38   38-42   42-46
train      32589    4832    4842    4847    4835    4753    4560    3920
val         9454    1408    1408    1406    1407    1390    1347    1088


## 2. Models

The same body for all three. The told model adds a shift per channel made from the QP; it starts at zero, so untrained it
is the blind model.

In [4]:
class Blind(torch.nn.Module):
    """A model that is not told, on loaders that tell: it drops the QP."""

    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, x, condition=None):
        return self.model(x)


def build(told):
    if told:
        return models.ToldLuma(conditions=1, width=WIDTH, blocks=BLOCKS, stem=STEM, scale=SCALE)
    return Blind(models.LumaUpscaler((WIDTH,), BLOCKS, STEM, scale=SCALE))


assert build(told=True).halo <= manifest["halo"], "the model reaches further than the patches' halo"

## 3. Train

Notebook 8's recipe: Adam from 1e-3, batch 32, `metrics.ssim_l1_loss`, the epoch with the best SSIM-Y kept, until it has
converged. The specialist trains and selects on QP 38 and up only. All three start from the same weights. Run the cell again
after an interruption and it carries on; a finished training just loads.

In [5]:
def train(name, told, low=0):
    """A fresh model trained on patches at QP `low` and up, or its saved training loaded."""
    torch.manual_seed(SEED)
    model = build(told).to(DEVICE)
    torch.manual_seed(SEED)
    history, best = training.train(model, batches(train_set, low, train=True), batches(val_set, low), trim, MAX_EPOCHS,
                                   learning_rate=LEARNING_RATE, loss_fn=metrics.ssim_l1_loss, select="ssim",
                                   checkpoint=RUN_DIR / f"{name}.pt", verbose=0)
    print(f"{name:<12} best epoch {best['epoch']:3} of {len(history):3}, ssim-y {best['score']:.4f}")
    return model.eval()


trained = {"blind": train("blind", told=False),
           "told QP": train("told_qp", told=True),
           "specialist": train("specialist", told=False, low=SPECIALIST_FROM)}

blind        best epoch 100 of 112, ssim-y 0.9077
told_qp      best epoch  91 of  97, ssim-y 0.9121
specialist   best epoch  69 of  93, ssim-y 0.8491


## 4. Gain over bicubic, by QP

Each cell is SSIM-Y and luma dB over bicubic on that bin's validation patches. The specialist is shown on every bin, but it
was only trained for QP 38 and up.

In [6]:
bicubic = Blind(models.Resize(SCALE))
print(f"{'QP':>7}{'patches':>9}{'bicubic':>9}" + "".join(f"{name:>20}" for name in trained))
print(f"{'':>25}" + "".join(f"{'ssim-y / dB':>20}" for _ in trained))
for low, high in BINS:
    loader = batches(val_set, low, high)
    base = training.evaluate(bicubic, loader, trim)
    cells = []
    for model in trained.values():
        scores = training.evaluate(model, loader, trim)
        cells.append(f"{scores['ssim'] - base['ssim']:+.4f} / {scores['y'] - base['y']:+.2f}")
    print(f"{f'{low}-{high}':>7}{len(loader.dataset):9}{base['ssim']:9.4f}" + "".join(f"{cell:>20}" for cell in cells))

     QP  patches  bicubic               blind             told QP          specialist
                                  ssim-y / dB         ssim-y / dB         ssim-y / dB
   0-18     1408   0.9568     +0.0058 / +0.29     +0.0149 / +2.20     -0.0407 / -4.35
  18-26     1408   0.9378     +0.0147 / +1.08     +0.0197 / +1.89     -0.0230 / -2.33
  26-30     1406   0.9096     +0.0251 / +1.39     +0.0279 / +1.67     -0.0026 / -0.89
  30-34     1407   0.8829     +0.0330 / +1.31     +0.0357 / +1.45     +0.0169 / +0.04
  34-38     1390   0.8434     +0.0369 / +1.04     +0.0401 / +1.12     +0.0329 / +0.56
  38-42     1347   0.8154     +0.0374 / +0.79     +0.0417 / +0.84     +0.0408 / +0.69
  42-46     1088   0.7999     +0.0360 / +0.69     +0.0401 / +0.75     +0.0404 / +0.73


## 5. What it says

To fill in after a run. How to read the table:

- **Specialist clearly above the told model at 38-46** (by more than seed luck): a separate high QP model pays.
- **Told model level with the specialist there, and at least level with the blind one elsewhere**: one told model.
- **Told model level with the blind one everywhere**: QP adds nothing; train the blind model on all QPs.

A told model ships as notebook 8's does, baked: a plain graph for each of a few QP values, and the client picks the nearest
for each frame's QP.